In [ ]:
# TODO @pszmk do plots for differentiation of mutations, first last 4/7 aa

In [1]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

In [2]:
AMINO_ACID_CLASSES = {
    "polarity": {
        "hydrophobic": ["A", "V", "L", "I", "M", "F", "W", "P", "G"],
        "polar": ["S", "T", "Y", "C", "N", "Q"],
        "charged": ["K", "R", "H", "D", "E"],
    },
    "charge": {
        "positive": ["K", "R", "H"],
        "negative": ["D", "E"],
        "neutral": [
            "A",
            "V",
            "L",
            "I",
            "M",
            "F",
            "W",
            "P",
            "G",
            "S",
            "T",
            "Y",
            "C",
            "N",
            "Q",
        ],
    },
    "chemical_type": {
        "aliphatic": ["G", "A", "V", "L", "I"],
        "aromatic": ["F", "Y", "W"],
        "hydroxyl": ["S", "T", "Y"],
        "acidic": ["D", "E"],
        "amide": ["N", "Q"],
        "basic": ["K", "R", "H"],
        "sulfur": ["C", "M"],
        "imino": ["P"],
        # "other": [],  # placeholder (no gaps)
    },
    "essentiality": {
        "essential": ["F", "V", "T", "W", "M", "L", "I", "K", "H"],
        "non_essential": ["A", "N", "D", "E", "Q", "G", "P", "S", "C", "Y"],
    },
    "aromaticity": {
        "aromatic": ["F", "Y", "W"],
        "non_aromatic": [
            "A",
            "R",
            "N",
            "D",
            "C",
            "E",
            "Q",
            "G",
            "H",
            "I",
            "L",
            "K",
            "M",
            "P",
            "S",
            "T",
            "V",
        ],
    },
    "side_chain_size": {
        "small": ["G", "A", "S", "T", "P"],
        "medium": ["C", "N", "D", "Q", "E"],
        "large": ["V", "L", "I", "M", "F", "Y", "W", "K", "R", "H"],
    },
}

GRAM_POSITIVE = [
    "S. aureus ATCC 12600",
    "S. aureus (ATCC BAA-1556) - MRSA",
    "vancomycin-resistant E. faecalis ATCC 700802",
    "vancomycin-resistant E. faecium ATCC 700221",
    "L. monocytogenes ATCC 19111 (BEIRES NR-106)",
    "C. aerofaciens ATCC25986",
    "C. scindens ATCC35704",
    "C. spiroforme ATCC29900",
    "E. rectale ATCC33656",
    "C. symbiosum",
    "R. obeum",
    "R. torques",
]

GRAM_NEGATIVE = [
    "A. baumannii ATCC 19606",
    "E. coli ATCC 11775",
    "E. coli AIG221",
    "E. coli AIG222",
    "K. pneumoniae ATCC 13883",
    "P. aeruginosa PA01",
    "P. aeruginosa PA14",
    "E. coli Nissle",
    "Salmonella enterica ATCC 9150 (BEIRES NR-515)",
    "Salmonella enterica (BEIRES NR-170)",
    "Salmonella enterica ATCC 9150 (BEIRES NR-174)",
    "A. muciniphila ATCC BAA-835",
    "B. fragilis ATCC25285",
    "B. vulgatus ATCC8482",
    "B. thetaiotaomicron ATCC29148",
    "B. thetaiotaomicron Complemmented",
    "B. thetaiotaomicron Mutant",
    "B. uniformis ATCC8492",
    "B. eggerthi ATCC27754",
    "B. ovatus ATCC8483",
    "P. distasonis ATCC8503",
    "P. copri DSMZ18205",
]

In [3]:
ALL_AA = sorted("ACDEFGHIKLMNPQRSTVWY")

In [4]:
import numpy as np
import pandas as pd
from typing import Callable


def aggregate_mutation_dict(
    data_dict: dict,
    value_key: str | None = None,
    subset: list[str] | None = None,
    agg_func: Callable | None = np.mean,
    agg_func_kwargs: dict | None = None,
) -> dict:
    """
    Aggregates mutation dictionaries and returns aggregated values.

    Parameters
    ----------
    data_dict : dict
        Full mutation dictionary with structure:
        { (from_aa, to_aa): { value_key: { variable_name: list_of_values } } }
    value_key : str, optional
        The inner key to aggregate over (e.g. "diff"). If None, assumes data_dict
        has structure { (from_aa, to_aa): { variable_name: list_of_values } }.
    subset : list of str, optional
        List of variable names to include (e.g. species names). If None, includes all.
    agg_func : callable, default np.mean
        Aggregation function to apply over the subset variables (e.g., np.mean, np.max,
        np.min, lambda x: np.quantile(x, 0.95), etc.).
    agg_func_kwargs : dict, optional
        Additional keyword arguments to pass to the aggregation function.

    Returns
    -------
    dict
        Dictionary with structure { (from_aa, to_aa): { 'mean': float, 'std': float, 'n': int } }
    """
    result = {}

    for (aa1, aa2), record in data_dict.items():
        # Handle optional value_key
        if value_key is not None:
            if value_key not in record:
                continue
            value_dict = record[value_key]
        else:
            value_dict = record

        all_values = []

        # include only selected variables (or all if subset is None)
        for var, vals in value_dict.items():
            if subset is None or var in subset:
                all_values.append(vals)

        if len(all_values) == 0:
            mean = np.nan
            std = np.nan
            n = 0
        else:
            arr = np.array(all_values, dtype=float)
            agg_arr = (
                agg_func(arr, **agg_func_kwargs if agg_func_kwargs else {})
                if agg_func
                else arr
            )
            assert (
                agg_arr.ndim == 1
            ), f"Aggregation function {agg_func} returned array with {agg_arr.ndim} dimensions"
            mean = float(np.mean(agg_arr))
            std = float(np.std(agg_arr))
            n = len(agg_arr)

        result[(aa1, aa2)] = {"mean": mean, "std": std, "n": n}

    return result


def aggregate_mutation_dict_to_df(
    data_dict: dict,
    value_key: str | None = None,
    subset: list[str] | None = None,  # REQUIRED
    agg_func: Callable | None = np.mean,
    agg_func_kwargs: dict | None = None,
):
    """
    Aggregates mutation dictionaries of the structure:
        { (from_aa, to_aa): { value_key: { variable_name: list_of_values } } }

    Parameters
    ----------
    data_dict : dict
        Full mutation dictionary.
    value_key : str, optional
        The inner key to aggregate over (e.g. "diff"). If None, assumes data_dict
        has structure { (from_aa, to_aa): { variable_name: list_of_values } }.
    subset : list of str, optional
        List of variable names to include (e.g. species names). If None, includes all.
    agg_func : callable, default np.mean
        Aggregation function to apply over the subset variables (e.g., np.mean, np.max,
        np.min, lambda x: np.quantile(x, 0.95), etc.).
    agg_func_kwargs : dict, optional
        Additional keyword arguments to pass to the aggregation function.

    Returns
    -------
    df_mean, df_std, df_n : pivot tables
        Pivot tables with aggregated values (using agg_func), std, and n.
    """
    # Call the internal aggregation function
    aggregated = aggregate_mutation_dict(
        data_dict=data_dict,
        value_key=value_key,
        subset=subset,
        agg_func=agg_func,
        agg_func_kwargs=agg_func_kwargs,
    )

    # Convert to DataFrame format
    rows = []
    for (aa1, aa2), stats in aggregated.items():
        rows.append(
            {
                "from_aa": aa1,
                "to_aa": aa2,
                "mean": stats["mean"],
                "std": stats["std"],
                "n": stats["n"],
            }
        )

    df = pd.DataFrame(rows)

    df_mean = df.pivot(index="from_aa", columns="to_aa", values="mean")
    df_std = df.pivot(index="from_aa", columns="to_aa", values="std")
    df_n = df.pivot(index="from_aa", columns="to_aa", values="n")

    return df_mean, df_std, df_n

In [5]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [6]:
def aa_mutation_heatmap(
    df_mean: pd.DataFrame,
    aa_list: list[str],
    df_std: pd.DataFrame | None = None,
    df_n: pd.DataFrame | None = None,
    AA_group: dict[str, list[str]] | None = None,
    cmap: str = "magma",
    nan_color: str = "black",
    figsize=(9, 7),
    fontsize: int = 11,
    title_fontsize: int = 12,
    label_fontsize: int = 11,
    title: str = "",
    x_label: str = "To AA n-gram",
    y_label: str = "From AA n-gram",
    group_linewidth: float = 1.5,
    group_colors: dict[str, str] | None = None,
    # NEW: separate axis control
    x_bar_gap: float = 0.0,  # heatmap → X-axis color bar
    x_label_gap: float = 0.0,  # color bar → X-axis label
    y_bar_gap: float = 0.0,  # heatmap → Y-axis color bar
    y_label_gap: float = 0.0,  # color bar → Y-axis label
    group_label_color: str = "black",
    # NEW: colorbar control
    vmin: float | None = None,  # minimum value for colorbar
    vmax: float | None = None,  # maximum value for colorbar
    cbar_shrink: float = 0.8,  # shrink factor for colorbar (0-1)
    cbar_aspect: float = 20,  # aspect ratio of colorbar
    cbar_pad: float = 0.02,  # padding between heatmap and colorbar
    # NEW: threshold highlighting
    df_threshold: (
        pd.DataFrame | None
    ) = None,  # dataframe with values to compare against threshold
    threshold: float | None = None,  # threshold value
    threshold_direction: str = "both",  # "above", "below", or "both"
    threshold_color: str = "red",  # color of threshold rectangle
    threshold_linewidth: float = 2.0,  # linewidth of threshold rectangle
):
    """
    Heatmap with fully symmetric and independently controlled
    X/Y axis group bars and group labels.
    """

    df = df_mean.copy()

    # === 1. Group sorting ===
    if AA_group is not None:
        sorted_aa = []
        group_sizes, group_names = [], []

        for gname, members in AA_group.items():
            present = [aa for aa in aa_list if aa in members]
            if present:
                sorted_aa.extend(present)
                group_sizes.append(len(present))
                group_names.append(gname)

        df = df.loc[sorted_aa, sorted_aa]
        if df_std is not None:
            df_std = df_std.loc[sorted_aa, sorted_aa]
        if df_n is not None:
            df_n = df_n.loc[sorted_aa, sorted_aa]
        if df_threshold is not None:
            df_threshold = df_threshold.loc[sorted_aa, sorted_aa]
    else:
        sorted_aa = aa_list
        group_sizes, group_names = [], []
        if df_threshold is not None:
            df_threshold = df_threshold.loc[sorted_aa, sorted_aa]

    # === 2. Annotation matrix ===
    annot = np.empty(df.shape, dtype=object)
    for i, r in enumerate(df.index):
        for j, c in enumerate(df.columns):
            if pd.isna(df.loc[r, c]):
                annot[i, j] = ""
                continue
            txt = f"{df.loc[r, c]:.2f}"
            if df_std is not None:
                sd = df_std.loc[r, c]
                if not pd.isna(sd):
                    txt += f"\n± {sd:.2f}"
            if df_n is not None:
                n = df_n.loc[r, c]
                if not pd.isna(n):
                    txt += f"\n(n={int(n)})"
            annot[i, j] = txt

    # === 3. NaN color ===
    base_cmap = plt.get_cmap(cmap).copy()
    base_cmap.set_bad(color=nan_color)
    mask = pd.isna(df)

    # === 4. Draw heatmap ===
    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        df,
        cmap=base_cmap,
        annot=annot,
        fmt="",
        mask=mask,
        square=True,
        cbar=True,
        vmin=vmin,
        vmax=vmax,
        cbar_kws={
            "shrink": cbar_shrink,
            "aspect": cbar_aspect,
            "pad": cbar_pad,
        },
        annot_kws={"fontsize": fontsize - 1},
        ax=ax,
    )

    ax.set_title(title, fontsize=title_fontsize)
    ax.set_xlabel(x_label, fontsize=label_fontsize)
    ax.set_ylabel(y_label, fontsize=label_fontsize)
    ax.tick_params(axis="both", labelsize=label_fontsize)

    # === 4.5. Threshold highlighting ===
    if df_threshold is not None and threshold is not None:
        # Determine which cells to highlight
        highlight_mask = pd.DataFrame(
            False, index=df_threshold.index, columns=df_threshold.columns
        )

        if threshold_direction == "above":
            highlight_mask = df_threshold > threshold
        elif threshold_direction == "below":
            highlight_mask = df_threshold < threshold
        elif threshold_direction == "both":
            highlight_mask = (df_threshold > threshold) | (df_threshold < threshold)
        else:
            raise ValueError(
                f"threshold_direction must be 'above', 'below', or 'both', got '{threshold_direction}'"
            )

        # Draw rectangles around highlighted cells
        # Note: heatmap cells are centered at integer positions (0.5, 1.5, 2.5, ...)
        for i, row_idx in enumerate(df_threshold.index):
            for j, col_idx in enumerate(df_threshold.columns):
                if highlight_mask.loc[row_idx, col_idx] and not pd.isna(
                    df_threshold.loc[row_idx, col_idx]
                ):
                    # Draw rectangle around the cell
                    # Cell boundaries: [i, i+1] x [j, j+1] in data coordinates
                    rect = plt.Rectangle(
                        (j, i),  # bottom-left corner
                        1,  # width
                        1,  # height
                        fill=False,
                        edgecolor=threshold_color,
                        linewidth=threshold_linewidth,
                        transform=ax.transData,
                    )
                    ax.add_patch(rect)

    # === 5. Group boundaries + bars + labels (AXES COORDINATES) ===
    if AA_group is not None and group_sizes:

        boundaries = np.cumsum(group_sizes)
        mids = boundaries - np.array(group_sizes) / 2
        total = len(sorted_aa)

        # Convert mid positions from data coords → axes coords
        mids_axes = mids / total
        size_axes = np.array(group_sizes) / total

        bar_thickness_axes = 0.02  # thickness as fraction of axis

        # Separator lines (still in data coordinates)
        for b in boundaries[:-1]:
            ax.axhline(b, color="black", lw=group_linewidth)
            ax.axvline(b, color="black", lw=group_linewidth)

        # Draw bars + labels in AXES COORDINATES
        for mid_ax, gname, gsize_ax in zip(mids_axes, group_names, size_axes):

            col = group_colors.get(gname, "lightgray") if group_colors else "lightgray"

            # === Y-axis color bar ===
            # Reverse Y-axis position to match heatmap order (top to bottom in data = bottom to top in axes)
            mid_ax_y_reversed = 1.0 - mid_ax
            ax.add_patch(
                plt.Rectangle(
                    (-y_bar_gap, mid_ax_y_reversed - gsize_ax / 2),
                    bar_thickness_axes,
                    gsize_ax,
                    transform=ax.transAxes,
                    clip_on=False,
                    color=col,
                )
            )

            # === Y-axis label ===
            ax.text(
                -y_bar_gap - bar_thickness_axes - y_label_gap,
                mid_ax_y_reversed,
                gname,
                ha="center",
                va="center",
                transform=ax.transAxes,
                rotation=90,
                fontsize=fontsize,
                color=group_label_color,
            )

            # === X-axis color bar (BOTTOM) ===
            ax.add_patch(
                plt.Rectangle(
                    (mid_ax - gsize_ax / 2, -x_bar_gap),
                    gsize_ax,
                    bar_thickness_axes,
                    transform=ax.transAxes,
                    clip_on=False,
                    color=col,
                )
            )

            # === X-axis label (BOTTOM) ===
            ax.text(
                mid_ax,
                -x_bar_gap - bar_thickness_axes - x_label_gap,
                gname,
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=fontsize,
                color=group_label_color,
            )

    fig.tight_layout()
    return fig, ax

In [7]:
from scipy.stats import mannwhitneyu, wilcoxon
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd
from typing import Callable


def mutation_mannwhitney_test(
    dict_a: dict,
    dict_b: dict | None = None,  # NEW: optional, if None test against zero
    value_key: str = None,
    subset: list[str] = None,  # REQUIRED
    alternative: str = "two-sided",
    agg_func: Callable = np.mean,  # NEW: aggregation function
    agg_func_kwargs: dict | None = None,  # NEW: kwargs for aggregation function
    # NEW: kwargs
    mannwhitney_kwargs: dict | None = None,  # passed into mannwhitneyu
    wilcoxon_kwargs: dict | None = None,  # passed into wilcoxon (when dict_b is None)
    multitest: bool = False,  # whether to apply BH correction
    multitest_kwargs: dict | None = None,  # kwargs passed into multipletests
):
    """
    Performs Mann–Whitney U test for each mutation (aa1 -> aa2)
    comparing values from dict_a vs dict_b.

    If dict_b is None, performs one-sample Wilcoxon signed-rank test
    against zero for values from dict_a.

    Supports optional multiple testing correction (Benjamini–Hochberg).

    Parameters
    ----------
    dict_a : dict
        Mutation dictionary.
    dict_b : dict or None
        Mutation dictionary. If None, tests dict_a values against zero.
    value_key : str
        Inner key to extract values from.
    subset : list[str]
        REQUIRED list of variable names to include.
    alternative : str
        two-sided, less, greater (Mann–Whitney argument)
    agg_func : callable, default np.mean
        Aggregation function to apply over the subset variables (e.g., np.mean, np.max,
        np.min, lambda x: np.quantile(x, 0.95), etc.).
    agg_func_kwargs : dict, optional
        Additional keyword arguments to pass to the aggregation function.
    mannwhitney_kwargs : dict
        Additional arguments forwarded to scipy.stats.mannwhitneyu.
    wilcoxon_kwargs : dict
        Additional arguments forwarded to scipy.stats.wilcoxon (when dict_b is None).
    multitest : bool
        Whether to apply multiple testing correction (Benjamini–Hochberg).
    multitest_kwargs : dict
        Extra keyword arguments passed to statsmodels.multipletests().

    Returns
    -------
    df_p_raw : pivot table of raw p-values
    df_p_adj : pivot table of corrected p-values (NaN if multitest=False)
    df_u      : pivot table of U statistics (or W statistics if one-sample)
    df_n1     : pivot table of sample counts in dict_a
    df_n2     : pivot table of sample counts in dict_b (NaN if one-sample)
    df_effect : pivot table of median difference (median(a) - median(b)) or median(a) if one-sample
    """

    if mannwhitney_kwargs is None:
        mannwhitney_kwargs = {}
    if wilcoxon_kwargs is None:
        wilcoxon_kwargs = {}
    if multitest_kwargs is None:
        multitest_kwargs = {}
    if agg_func_kwargs is None:
        agg_func_kwargs = {}

    rows = []
    one_sample = dict_b is None

    # union of all mutation pairs
    if one_sample:
        all_keys = set(dict_a.keys())
    else:
        all_keys = set(dict_a.keys()).union(dict_b.keys())

    for aa1, aa2 in all_keys:

        def extract_values(rec):
            if rec is None or value_key not in rec:
                return []
            all_values = []
            for var, vals in rec[value_key].items():
                if var in subset:
                    all_values.append(vals)

            if len(all_values) == 0:
                return []

            arr = np.array(all_values, dtype=float)
            agg_arr = agg_func(arr, **agg_func_kwargs)
            assert (
                agg_arr.ndim == 1
            ), f"Aggregation function {agg_func} returned array with {agg_arr.ndim} dimensions"
            return list(agg_arr)

        vals_a = extract_values(dict_a.get((aa1, aa2)))

        if one_sample:
            # One-sample test against zero
            if len(vals_a) == 0:
                w_stat = np.nan
                p_val = np.nan
                effect = np.nan
            else:
                try:
                    w_stat, p_val = wilcoxon(
                        vals_a, alternative=alternative, **wilcoxon_kwargs
                    )
                    effect = np.median(vals_a)  # median value (difference from zero)
                except Exception:
                    w_stat = np.nan
                    p_val = np.nan
                    effect = np.nan

            rows.append(
                {
                    "from_aa": aa1,
                    "to_aa": aa2,
                    "p_raw": p_val,
                    "u": w_stat,  # storing W statistic in 'u' column for consistency
                    "n1": len(vals_a),
                    "n2": np.nan,
                    "effect": effect,
                }
            )
        else:
            # Two-sample test
            vals_b = extract_values(dict_b.get((aa1, aa2)))

            # if one side has zero samples → cannot test
            if len(vals_a) == 0 or len(vals_b) == 0:
                u_stat = np.nan
                p_val = np.nan
                effect = np.nan
            else:
                try:
                    u_stat, p_val = mannwhitneyu(
                        vals_a, vals_b, alternative=alternative, **mannwhitney_kwargs
                    )
                    effect = np.median(vals_a) - np.median(vals_b)

                except Exception:
                    u_stat = np.nan
                    p_val = np.nan
                    effect = np.nan

            rows.append(
                {
                    "from_aa": aa1,
                    "to_aa": aa2,
                    "p_raw": p_val,
                    "u": u_stat,
                    "n1": len(vals_a),
                    "n2": len(vals_b),
                    "effect": effect,
                }
            )

    df = pd.DataFrame(rows)

    # --- Multiple testing correction ---
    if multitest:
        pvals = df["p_raw"].values
        _, p_adj, _, _ = multipletests(
            pvals, method="fdr_bh", **multitest_kwargs  # Benjamini-Hochberg
        )
        df["p_adj"] = p_adj
    else:
        df["p_adj"] = np.nan

    # --- Make pivot tables ---
    df_p_raw = df.pivot(index="from_aa", columns="to_aa", values="p_raw")
    df_p_adj = df.pivot(index="from_aa", columns="to_aa", values="p_adj")
    df_u = df.pivot(index="from_aa", columns="to_aa", values="u")
    df_n1 = df.pivot(index="from_aa", columns="to_aa", values="n1")
    df_n2 = df.pivot(index="from_aa", columns="to_aa", values="n2")
    df_effect = df.pivot(index="from_aa", columns="to_aa", values="effect")

    return df_p_raw, df_p_adj, df_u, df_n1, df_n2, df_effect

---

In [8]:
diff_values_path = Path(
    "/home/prz/bioml/pep-compass/results/mutation_analysis/hydramp_x_ampsphere/direction_threshold=0.001_token_threshold=0.1_jacobian_mode=approx_jacobian_eps=0.05/ngram1/diff_values.csv"
)
diff_df = pd.read_csv(diff_values_path)

/tmp/ipykernel_5986/2146905077.py:4: DtypeWarning: Columns (39,74,82,83,84,85,86) have mixed types. Specify dtype option on import or set low_memory=False.
  diff_df = pd.read_csv(diff_values_path)


In [9]:
from scripts.mutation_analysis.analysis import (
    compute_mutation_statistics_from_df,
    compute_ranks_for_single_sample,
)
from scripts.mutation_analysis.mutations import compute_aggregate_mutation_counter

---

In [10]:
group_colors = {
    "hydrophobic": "#F28E8E",
    "polar": "#8EC9F2",
    "charged": "#A9E68E",
}

#### 1-grams

In [11]:
aggregate_counter_ngram1 = compute_aggregate_mutation_counter(
    diff_df=diff_df,
    parent_col="parent",
    mutant_col="mutant",
    ngram_size=1,
    allow_ngrams_overlap=True,
)

mutation_statistics_ngram1 = compute_mutation_statistics_from_df(
    diff_df=diff_df,
    aggregate_counter=aggregate_counter_ngram1,
    parent_col="parent",
    mutant_col="mutant",
    value_cols=[f"{b}_log2_diff" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
    ngram_size=1,
    allow_ngrams_overlap=True,
)

In [ ]:
def aa_mutation_heatmap_factory_for_some_preset(subset, title, mutation_statistics):
    transition_mean, transition_std, transition_n = aggregate_mutation_dict_to_df(
        mutation_statistics,
        value_key="diff",
        subset=subset,
        agg_func=np.mean,
        agg_func_kwargs={"axis": 0},
    )

    df_p_raw, df_p_adj, df_u, df_n1, df_n2, df_effect = mutation_mannwhitney_test(
        dict_a=mutation_statistics,
        value_key="diff",
        subset=subset,
        agg_func=np.mean,  # matches aggregate_mutation_dict_to_df
        agg_func_kwargs={"axis": 0},
        multitest=True,
    )

    return aa_mutation_heatmap(
        df_mean=transition_mean,
        aa_list=ALL_AA,
        df_std=transition_std,
        df_n=transition_n,
        df_threshold=df_p_adj,
        threshold=0.001,
        threshold_direction="below",
        threshold_color="blue",
        AA_group=AMINO_ACID_CLASSES["polarity"],
        cmap="RdYlGn_r",
        nan_color="white",
        figsize=(24, 24),
        fontsize=11,
        title=title,
        x_label="To AA n-gram",
        y_label="From AA n-gram",
        group_linewidth=4,
        group_colors=group_colors,
        x_bar_gap=0.07,
        y_bar_gap=0.07,
        title_fontsize=30,
        label_fontsize=18,
    )

In [ ]:
aa_mutation_heatmap_factory_for_some_preset(
    subset=[f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
    title="All strains | log2 diff | 1-gram",
    mutation_statistics=mutation_statistics_ngram1,
)

In [ ]:
aa_mutation_heatmap_factory_for_some_preset(
    subset=[f"{b}_log2" for b in GRAM_POSITIVE],
    title="Gram + | log2 diff | 1-gram",
    mutation_statistics=mutation_statistics_ngram1,
)

In [ ]:
aa_mutation_heatmap_factory_for_some_preset(
    subset=[f"{b}_log2" for b in GRAM_NEGATIVE],
    title="Gram - | log2 diff | 1-gram",
    mutation_statistics=mutation_statistics_ngram1,
)

### Ranks

In [ ]:
ranks_ngram1 = compute_ranks_for_single_sample(
    stats_dict=mutation_statistics_ngram1,
    aggregate_cols=[f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
    aggregate_counter=aggregate_counter_ngram1,
    aggregation_func="mean",
)
gp_ranks_ngram1 = compute_ranks_for_single_sample(
    stats_dict=mutation_statistics_ngram1,
    aggregate_cols=[f"{b}_log2" for b in GRAM_POSITIVE],
    aggregate_counter=aggregate_counter_ngram1,
    aggregation_func="mean",
)
gn_ranks_ngram1 = compute_ranks_for_single_sample(
    stats_dict=mutation_statistics_ngram1,
    aggregate_cols=[f"{b}_log2" for b in GRAM_NEGATIVE],
    aggregate_counter=aggregate_counter_ngram1,
    aggregation_func="mean",
)

In [ ]:
def aa_mutation_heatmap_factory_for_ranks_relative_for_some_preset(
    subset, reference_subset, title, mutation_statistics, reference_mutation_statistics
):

    # Aggregate mutation statistics for the subset
    aggregated_mutation_stats = aggregate_mutation_dict(
        data_dict=mutation_statistics,
        value_key="diff",
        subset=subset,
        agg_func=np.mean,
        agg_func_kwargs={"axis": 0},
    )

    # Aggregate reference mutation statistics for the subset
    aggregated_reference_mutation_stats = aggregate_mutation_dict(
        data_dict=reference_mutation_statistics,
        value_key="diff",
        subset=reference_subset,
        agg_func=np.mean,
        agg_func_kwargs={"axis": 0},
    )

    ranks = compute_ranks_for_single_sample(
        stats_dict=aggregated_mutation_stats,
        aggregate_cols=["mean"],
        aggregate_counter=aggregate_counter_ngram1,
        aggregation_func=None,
    )
    reference_ranks = compute_ranks_for_single_sample(
        stats_dict=aggregated_reference_mutation_stats,
        aggregate_cols=["mean"],
        aggregate_counter=aggregate_counter_ngram1,
        aggregation_func=None,
    )

    # Subtract reference ranks from ranks
    comp_ranks = {}
    for (from_aa, to_aa), rank_value in ranks.items():
        reference_value = reference_ranks.get((from_aa, to_aa))
        comp_ranks[(from_aa, to_aa)] = {}
        comp_ranks[(from_aa, to_aa)]["mean_rank_diff"] = (
            rank_value["mean_rank"] - reference_value["mean_rank"]
        )

    transition_mean, transition_std, transition_n = aggregate_mutation_dict_to_df(
        comp_ranks, subset=["mean_rank_diff"], agg_func=None
    )

    return aa_mutation_heatmap(
        df_mean=transition_mean,
        aa_list=ALL_AA,
        df_std=None,
        df_n=transition_n,
        threshold_direction="below",
        threshold_color="blue",
        AA_group=AMINO_ACID_CLASSES["polarity"],
        cmap="RdYlGn_r",
        nan_color="white",
        figsize=(24, 24),
        fontsize=11,
        title=title,
        x_label="To AA n-gram",
        y_label="From AA n-gram",
        group_linewidth=4,
        group_colors=group_colors,
        x_bar_gap=0.07,
        y_bar_gap=0.07,
        title_fontsize=30,
        label_fontsize=18,
    )

In [ ]:
aa_mutation_heatmap_factory_for_ranks_relative_for_some_preset(
    subset=[f"{b}_log2" for b in GRAM_POSITIVE],
    reference_subset=[f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
    title="Gram + | ranks rel to population | 1-gram",
    mutation_statistics=mutation_statistics_ngram1,
    reference_mutation_statistics=mutation_statistics_ngram1,
)

In [ ]:
# right no of samples
aa_mutation_heatmap_factory_for_ranks_relative_for_some_preset(
    subset=[f"{b}_log2" for b in GRAM_NEGATIVE],
    reference_subset=[f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
    title="Gram - | ranks rel to population | 1-gram",
    mutation_statistics=mutation_statistics_ngram1,
    reference_mutation_statistics=mutation_statistics_ngram1,
)

---

### 3-grams

In [11]:
import pickle

# aggregate_counter_ngram1 = compute_aggregate_mutation_counter(
#     diff_df=diff_df,
#     parent_col="parent",
#     mutant_col="mutant",
#     ngram_size=1,
#     allow_ngrams_overlap=True,
# )

# with open(
#     "/home/prz/bioml/pep-compass/results/nbgeneratedobjects/122025-mutation-signature-analysis/aggregate_counter_ngram3.pkl",
#     "wb",
# ) as f:
#     pickle.dump(aggregate_counter_ngram3, f)


with open(
    "/home/prz/bioml/pep-compass/results/nbgeneratedobjects/122025-mutation-signature-analysis/aggregate_counter_ngram3.pkl",
    "rb",
) as f:
    aggregate_counter_ngram3 = pickle.load(f)

In [12]:
agg_diff_df = diff_df[["mutant", "position", "parent", "parent_id"]].copy()
agg_diff_df["log2_diff_mean"] = (
    diff_df[[f"{b}_log2_diff" for b in GRAM_POSITIVE + GRAM_NEGATIVE]]
    .mean(axis=1)
    .values
)
agg_diff_df["GRAM_POSITIVE_log2_diff_mean"] = (
    diff_df[[f"{b}_log2_diff" for b in GRAM_POSITIVE]].mean(axis=1).values
)
agg_diff_df["GRAM_NEGATIVE_log2_diff_mean"] = (
    diff_df[[f"{b}_log2_diff" for b in GRAM_NEGATIVE]].mean(axis=1).values
)

In [ ]:
mutation_statistics_ngram3 = compute_mutation_statistics_from_df(
    diff_df=agg_diff_df,
    aggregate_counter=aggregate_counter_ngram3,
    parent_col="parent",
    mutant_col="mutant",
    value_cols=["log2_diff_mean"],
    ngram_size=3,
    allow_ngrams_overlap=True,
)

---

### old Check out 3-grams

In [ ]:
from tqdm import tqdm
from collections import Counter

from scripts.mutation_analysis.mutations import process_bootstrap_mutations


def df_to_mutation_transition(
    diff_df: pd.DataFrame,
    # aggregate_counter: Counter,
    parent_col: str,
    mutant_col: str,
    value_cols: list[str],
    ngram_size: int,
    allow_ngrams_overlap: bool = True,
    use_pbar: bool = True,
) -> dict:
    """
    Compute mutation statistics dictionary from a single DataFrame.

    This function processes a DataFrame and returns a dict mapping mutation keys
    to dictionaries with column names as keys and lists of values as values.

    Args:
        diff_df: DataFrame to process (typically a single bootstrap sample)
        aggregate_counter: Aggregate Counter with all mutation keys
        parent_col: Column name containing parent sequences
        mutant_col: Column name containing mutant sequences
        value_cols: List of column names to collect statistics for
        ngram_size: Size of n-grams for mutation analysis
        allow_ngrams_overlap: Whether to allow overlapping n-grams

    Returns:
        Dict mapping mutation keys (tuples) to dicts with structure:
        {mutation_key: {col_name: [values]}}
    """
    aggregate_keys = set(aggregate_counter.keys())

    # Process mutations
    bootstrap_counters = process_bootstrap_mutations(
        bootstrap_df=diff_df,
        parent_col=parent_col,
        mutant_col=mutant_col,
        aggregate_mutation_keys=aggregate_keys,
        ngram_size=ngram_size,
        allow_ngrams_overlap=allow_ngrams_overlap,
    )

    # Initialize result dict: mutation_key -> {col_name: [values]}
    result: dict[tuple[str, str], dict[str, list]] = {}

    for key in aggregate_keys:
        result[key] = {}
        for col in value_cols:
            result[key][col] = []

    # Collect values for each row in DataFrame
    if use_pbar:
        pbar = tqdm(
            diff_df.iterrows(),
            total=len(diff_df),
            postfix={"rows": 0},
            desc="Computing mutation statistics",
        )
        iterator = pbar
        row_count = 0
    else:
        iterator = diff_df.iterrows()

    for idx, row in iterator:
        parent = row[parent_col]
        mutant = row[mutant_col]

        # Skip if either is NaN
        if pd.isna(parent) or pd.isna(mutant):
            continue

        pair_key = (parent, mutant)

        # Get counter for this pair
        counter = bootstrap_counters.get(pair_key, Counter())

        # For each mutation in this row, add the column values
        for mutation_key, count in counter.items():
            if count > 0:  # Only if mutation is present
                for col in value_cols:
                    # Append the value 'count' times (once per occurrence)
                    result[mutation_key][col].extend([row[col]] * count)

        # Update progress bar
        if use_pbar:
            row_count += 1
            pbar.set_postfix({"rows": row_count})

    return result

In [ ]:
# aggregate_counter_ngram3 = compute_aggregate_mutation_counter(
#     diff_df=diff_df,
#     parent_col="parent",
#     mutant_col="mutant",
#     ngram_size=3,
#     allow_ngrams_overlap=True,
#     use_pbar=True,
# )

In [30]:
import pickle

# with open(
#     "/home/prz/bioml/pep-compass/results/nbgeneratedobjects/122025-mutation-signature-analysis/aggregate_counter_ngram3.pkl",
#     "wb",
# ) as f:
#     pickle.dump(aggregate_counter_ngram3, f)


with open(
    "/home/prz/bioml/pep-compass/results/nbgeneratedobjects/122025-mutation-signature-analysis/aggregate_counter_ngram3.pkl",
    "rb",
) as f:
    aggregate_counter_ngram3 = pickle.load(f)

In [31]:
diff_df.columns

Index(['mutant', 'position', 'parent', 'parent_id',
       'A. baumannii ATCC 19606_mutants', 'E. coli ATCC 11775_mutants',
       'E. coli AIG221_mutants', 'E. coli AIG222_mutants',
       'K. pneumoniae ATCC 13883_mutants', 'P. aeruginosa PA01_mutants',
       ...
       'E. coli Nissle_log2_diff', 'E. coli Nissle_log2_diff_relative',
       'Salmonella enterica ATCC 9150 (BEIRES NR-515)_log2_diff',
       'Salmonella enterica ATCC 9150 (BEIRES NR-515)_log2_diff_relative',
       'Salmonella enterica (BEIRES NR-170)_log2_diff',
       'Salmonella enterica (BEIRES NR-170)_log2_diff_relative',
       'Salmonella enterica ATCC 9150 (BEIRES NR-174)_log2_diff',
       'Salmonella enterica ATCC 9150 (BEIRES NR-174)_log2_diff_relative',
       'L. monocytogenes ATCC 19111 (BEIRES NR-106)_log2_diff',
       'L. monocytogenes ATCC 19111 (BEIRES NR-106)_log2_diff_relative'],
      dtype='object', length=157)

In [32]:
agg_diff_df = diff_df[["mutant", "position", "parent", "parent_id"]].copy()
agg_diff_df["log2_diff_mean"] = (
    diff_df[[f"{b}_log2_diff" for b in GRAM_POSITIVE + GRAM_NEGATIVE]]
    .mean(axis=1)
    .values
)
agg_diff_df["GRAM_POSITIVE_log2_diff_mean"] = (
    diff_df[[f"{b}_log2_diff" for b in GRAM_POSITIVE]].mean(axis=1).values
)
agg_diff_df["GRAM_NEGATIVE_log2_diff_mean"] = (
    diff_df[[f"{b}_log2_diff" for b in GRAM_NEGATIVE]].mean(axis=1).values
)

In [34]:
mutation_statistics_ngram1 = compute_mutation_statistics_from_df(
    diff_df=diff_df,
    aggregate_counter=aggregate_counter_ngram3,
    parent_col="parent",
    mutant_col="mutant",
    value_cols=[
        "log2_mean_diff",
        "GRAM_POSITIVE_log2_mean_diff",
        "GRAM_NEGATIVE_log2_mean_diff",
    ],
    ngram_size=3,
    allow_ngrams_overlap=True,
)

KeyboardInterrupt: 

---

In [12]:
from scripts.mutation_analysis.mutations import (
    get_mutation_counts_for_substitution_mutations,
)

In [10]:
diff_df.head()

,mutant,position,parent,parent_id,A. baumannii ATCC 19606_mutants,E. coli ATCC 11775_mutants,E. coli AIG221_mutants,E. coli AIG222_mutants,K. pneumoniae ATCC 13883_mutants,P. aeruginosa PA01_mutants,...,E. coli Nissle_log2_diff,E. coli Nissle_log2_diff_relative,Salmonella enterica ATCC 9150 (BEIRES NR-515)_log2_diff,Salmonella enterica ATCC 9150 (BEIRES NR-515)_log2_diff_relative,Salmonella enterica (BEIRES NR-170)_log2_diff,Salmonella enterica (BEIRES NR-170)_log2_diff_relative,Salmonella enterica ATCC 9150 (BEIRES NR-174)_log2_diff,Salmonella enterica ATCC 9150 (BEIRES NR-174)_log2_diff_relative,L. monocytogenes ATCC 19111 (BEIRES NR-106)_log2_diff,L. monocytogenes ATCC 19111 (BEIRES NR-106)_log2_diff_relative
0,FLGLLFHGVHHVGKWIHGNIHGHH,18,FLGLLFHGVHHVGKWIHGLIHGHH,DBAASP_1001,83.673540,113.194336,105.330060,76.993530,59.806360,55.607685,...,-0.082164,-0.008496,0.383081,0.082083,0.178596,0.018772,0.184985,0.023641,0.330948,0.064074
1,RWRWPIRRKPIRPPWYP,8,RWRWPIRRPPIRPPWYP,DBAASP_14468,40.590380,55.146930,52.983055,59.419037,18.918674,16.940615,...,-0.365075,-0.043889,-0.697049,-0.157693,-0.281444,-0.033377,-0.148279,-0.018917,-0.437211,-0.085761
2,GLLKLIKHLL,7,GLLKLIKTLL,DBAASP_3136,22.504032,54.032513,64.881610,71.851280,14.132823,13.810359,...,-0.360050,-0.038516,-0.345010,-0.073766,-0.325616,-0.038441,-0.399434,-0.049090,-0.270553,-0.054349
3,KWCFRVCARAICYRRCR,9,KWCFRVCARGICYRRCR,DBAASP_11737,37.541595,63.494347,82.600550,75.528010,35.548100,36.541615,...,0.166954,0.016897,-0.203289,-0.044302,0.303532,0.035412,0.296614,0.036055,-0.195413,-0.036086
4,LKLKKKFKCLLLKKLLL,6,LKLKKKCKCLLLKKLLL,DBAASP_12052,7.523683,22.240595,15.238005,68.358574,14.288968,9.676406,...,0.318358,0.036298,-0.232400,-0.060535,0.598396,0.073187,0.262997,0.032668,-0.169435,-0.037343


In [14]:
get_mutation_counts_for_substitution_mutations(
    parent="FLGLLFHGVHHVGKWIHGLIHGHH",
    mutant="FLGLLFHGVHHVGKWIHGNIHGHH",
    ngram_size=3,
    allow_ngrams_overlap=True,
    aggregate_over_positions=True,
)

Counter({('HGL', 'HGN'): 1, ('GLI', 'GNI'): 1, ('LIH', 'NIH'): 1})

In [18]:
from tqdm import tqdm


def compute_mutation_statistics_from_df(
    diff_df: pd.DataFrame,
    parent_col: str,
    mutant_col: str,
    value_cols: list[str],
    ngram_size: int,
    allow_ngrams_overlap: bool = True,
    use_pbar: bool = False,
    return_samples: bool = True,
    return_mean: bool = False,
    compute_std: bool = False,
) -> dict:
    """
    Compute mutation statistics dictionary from a single DataFrame.

    This function processes a DataFrame and returns a dict mapping mutation keys
    to dictionaries with column names as keys and lists of values as values.
    Mutation keys are discovered automatically from the data.

    Args:
        diff_df: DataFrame to process (typically a single bootstrap sample)
        parent_col: Column name containing parent sequences
        mutant_col: Column name containing mutant sequences
        value_cols: List of column names to collect statistics for
        ngram_size: Size of n-grams for mutation analysis
        allow_ngrams_overlap: Whether to allow overlapping n-grams
        use_pbar: Whether to show progress bar
        return_samples: If True, store individual sample values in lists
        return_mean: If True, compute and return mean values (using running mean)
        compute_std: If True, compute and return standard deviation (requires return_mean=True)

    Returns:
        Dict mapping mutation keys (tuples) to dicts with structure:
        {mutation_key: {col_name: [values] | col_name_mean: float | col_name_std: float}}
    """
    # Validate parameters
    if compute_std and not return_mean:
        raise ValueError("compute_std requires return_mean=True")

    # Initialize result dict: mutation_key -> {col_name: ...}
    # Keys will be discovered as we process the data
    result: dict[tuple[str, str], dict[str, list | float]] = {}

    # For running mean/std computation
    running_means: dict[tuple[str, str], dict[str, tuple[float, int]]] = (
        {}
    )  # (mean, count)
    running_vars: dict[tuple[str, str], dict[str, tuple[float, int]]] = (
        {}
    )  # (M2, count) for Welford's algorithm

    # First pass: Collect values and/or compute running means
    if use_pbar:
        pbar = tqdm(
            diff_df.iterrows(),
            total=len(diff_df),
            postfix={"rows": 0},
            desc="Computing mutation statistics (pass 1)",
        )
        iterator = pbar
        row_count = 0
    else:
        iterator = diff_df.iterrows()

    for idx, row in iterator:
        parent = row[parent_col]
        mutant = row[mutant_col]

        # Skip if either is NaN
        if pd.isna(parent) or pd.isna(mutant):
            continue

        # Compute counter for this row directly
        counter = get_mutation_counts_for_substitution_mutations(
            parent=str(parent),
            mutant=str(mutant),
            ngram_size=ngram_size,
            allow_ngrams_overlap=allow_ngrams_overlap,
            aggregate_over_positions=True,
        )

        # For each mutation in this row, process the column values
        for mutation_key, count in counter.items():
            if count > 0:  # Only if mutation is present
                # Initialize mutation_key entry if not seen before
                if mutation_key not in result:
                    result[mutation_key] = {}
                    if return_mean:
                        running_means[mutation_key] = {}
                    if compute_std:
                        running_vars[mutation_key] = {}
                    for col in value_cols:
                        if return_samples:
                            result[mutation_key][col] = []
                        if return_mean:
                            running_means[mutation_key][col] = (0.0, 0)  # (mean, count)
                        if compute_std:
                            running_vars[mutation_key][col] = (0.0, 0)  # (M2, count)

                for col in value_cols:
                    value = row[col]

                    # Skip NaN values
                    if pd.isna(value):
                        continue

                    # Store samples if requested
                    if return_samples:
                        result[mutation_key][col].extend([value] * count)

                    # Running mean using Welford's online algorithm
                    if return_mean:
                        for _ in range(count):
                            current_mean, current_count = running_means[mutation_key][
                                col
                            ]
                            new_count = current_count + 1
                            new_mean = current_mean + (value - current_mean) / new_count
                            running_means[mutation_key][col] = (new_mean, new_count)

        # Update progress bar
        if use_pbar:
            row_count += 1
            pbar.set_postfix({"rows": row_count})

    # Store means
    if return_mean:
        for mutation_key in result.keys():
            for col in value_cols:
                mean_val, count = running_means[mutation_key][col]
                if count > 0:
                    result[mutation_key][f"{col}_mean"] = mean_val
                else:
                    result[mutation_key][f"{col}_mean"] = np.nan

    # Second pass: Compute standard deviation if requested
    if compute_std:
        if use_pbar:
            pbar = tqdm(
                diff_df.iterrows(),
                total=len(diff_df),
                postfix={"rows": 0},
                desc="Computing mutation statistics (pass 2: std)",
            )
            iterator = pbar
            row_count = 0
        else:
            iterator = diff_df.iterrows()

        for idx, row in iterator:
            parent = row[parent_col]
            mutant = row[mutant_col]

            # Skip if either is NaN
            if pd.isna(parent) or pd.isna(mutant):
                continue

            # Compute counter for this row directly
            counter = get_mutation_counts_for_substitution_mutations(
                parent=str(parent),
                mutant=str(mutant),
                ngram_size=ngram_size,
                allow_ngrams_overlap=allow_ngrams_overlap,
                aggregate_over_positions=True,
            )

            for mutation_key, count in counter.items():
                # Only process mutations we've seen in the first pass
                if mutation_key not in result:
                    continue
                if count > 0:
                    for col in value_cols:
                        value = row[col]

                        if pd.isna(value):
                            continue

                        mean_value = result[mutation_key].get(f"{col}_mean", np.nan)
                        if np.isnan(mean_value):
                            continue

                        # Running variance: accumulate squared differences from mean
                        for _ in range(count):
                            M2, n = running_vars[mutation_key][col]
                            n += 1
                            diff = value - mean_value
                            M2 += diff * diff
                            running_vars[mutation_key][col] = (M2, n)

            # Update progress bar
            if use_pbar:
                row_count += 1
                pbar.set_postfix({"rows": row_count})

        # Compute standard deviations
        for mutation_key in result.keys():
            for col in value_cols:
                M2, n = running_vars[mutation_key][col]
                if n > 1:
                    variance = M2 / n
                    std_value = np.sqrt(variance)
                    result[mutation_key][f"{col}_std"] = std_value
                elif n == 1:
                    result[mutation_key][f"{col}_std"] = 0.0
                else:
                    result[mutation_key][f"{col}_std"] = np.nan

    return result

In [ ]:
results = compute_mutation_statistics_from_df(
    diff_df=agg_diff_df,
    parent_col="parent",
    mutant_col="mutant",
    value_cols=[
        "log2_mean_diff",
        "GRAM_POSITIVE_log2_mean_diff",
        "GRAM_NEGATIVE_log2_mean_diff",
    ],
    ngram_size=3,
    allow_ngrams_overlap=True,
    use_pbar=True,
    return_samples=False,
    return_mean=True,
    compute_std=False,
)

Computing mutation statistics (pass 1):   0%|          | 0/303273 [00:00<?, ?it/s, rows=0]

Computing mutation statistics (pass 1):   0%|          | 0/303273 [00:16<?, ?it/s, rows=0]


KeyError: 'log2_diff_mean'

In [ ]:
# df_to_mutation_transition(
#     diff_df=diff_df,
#     aggregate_counter=aggregate_counter_ngram3,
#     parent_col="parent",
#     mutant_col="mutant",
#     value_cols=[
#         "log2_diff_mean",
#         "GRAM_POSITIVE_log2_diff_mean",
#         "GRAM_NEGATIVE_log2_diff_mean",
#     ],
#     ngram_size=3,
#     allow_ngrams_overlap=True,
#     use_pbar=True,
# )

In [ ]:
# mutation_statistics_ngram3 = compute_mutation_statistics_from_df(
#     diff_df=diff_df,
#     aggregate_counter=aggregate_counter_ngram3,
#     parent_col="parent",
#     mutant_col="mutant",
#     value_cols=[f"{b}_log2_diff" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
#     ngram_size=3,
#     allow_ngrams_overlap=True,
#     use_pbar=True,
#     return_samples=False,
#     return_mean=True,
#     compute_std=False,
# )

# With aggregate_counter but still memory-efficient with aggregation
mutation_statistics_ngram3 = compute_mutation_statistics_from_df(
    diff_df=diff_df,
    aggregate_counter=aggregate_counter_ngram3,
    parent_col="parent",
    mutant_col="mutant",
    value_cols=[f"{b}_log2_diff" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
    ngram_size=3,
    allow_ngrams_overlap=True,
    use_pbar=True,
    return_samples=False,  # Don't store individual samples
    return_mean=True,  # Compute mean using aggregate_counter
    compute_std=False,
    # Column aggregation
    agg_cols_subset=[
        f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE
    ],  # Clean column names (without _log2_diff)
    agg_func=np.mean,
)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Callable


def mutation_boxplot_groups(
    group_dicts: dict[str, dict],
    value_key: str,
    subset: list[str],
    test_group: str,
    reference_group: str | None = None,  # if None → test vs zero
    min_n: int = 5,  # minimum n in test_group (and ref if given)
    alternative: str = "two-sided",
    agg_func: Callable = np.mean,
    agg_func_kwargs: dict | None = None,
    mannwhitney_kwargs: dict | None = None,
    wilcoxon_kwargs: dict | None = None,
    multitest: bool = False,
    multitest_kwargs: dict | None = None,
    max_mutations: int | None = None,  # optionally keep only top-k by p-value
    figsize: tuple[float, float] = (12, 6),
    palette: dict[str, str] | None = None,
    title: str = "",
    ylabel: str = "Value",
    xlabel: str = "Mutation (from→to)",
):
    """
    Create sorted boxplots of mutation distributions across multiple groups,
    sorted by the p-value of a statistical comparison for a chosen test group.

    Parameters
    ----------
    group_dicts : dict[str, dict]
        Mapping group_name -> mutation dictionary with structure
        { (from_aa, to_aa): { value_key: { variable_name: list_of_values } } }
        (same as in `mutation_mannwhitney_test` and `aggregate_mutation_dict`).
    value_key : str
        Inner key to extract values from (e.g. "diff").
    subset : list[str]
        REQUIRED list of variable names to include (e.g. species names).
    test_group : str
        Name of the group to use for statistical testing and sorting.
    reference_group : str or None
        If provided, compare test_group vs reference_group (Mann–Whitney U).
        If None, compare test_group against zero (Wilcoxon signed-rank).
    min_n : int
        Minimum sample size required for the test_group (and reference_group if given),
        otherwise the mutation is dropped.
    alternative : str
        "two-sided", "less", or "greater" (passed to scipy tests).
    agg_func : callable, default np.mean
        Aggregation over the `subset` dimension, same semantics as in your helpers.
    agg_func_kwargs : dict, optional
        Extra kwargs for agg_func.
    mannwhitney_kwargs : dict, optional
        Extra kwargs for scipy.stats.mannwhitneyu.
    wilcoxon_kwargs : dict, optional
        Extra kwargs for scipy.stats.wilcoxon (when reference_group is None).
    multitest : bool
        Whether to apply Benjamini–Hochberg correction.
    multitest_kwargs : dict, optional
        Extra kwargs for statsmodels.multipletests.
    max_mutations : int or None
        If not None, keep only this many mutations with the smallest p-values.
    figsize : tuple
        Figure size.
    palette : dict[group_name, color] or None
        Optional custom colors per group.
    title, ylabel, xlabel : str
        Plot labels.

    Returns
    -------
    df_long : pd.DataFrame
        Long-format table used for plotting (mutation, group, value, p_raw, etc.).
    df_p : pd.DataFrame
        Wide p-value table (index=from_aa, columns=to_aa) for the test_group.
    fig, ax : matplotlib Figure and Axes
    """

    from scipy.stats import mannwhitneyu, wilcoxon
    from statsmodels.stats.multitest import multipletests

    if agg_func_kwargs is None:
        agg_func_kwargs = {}
    if mannwhitney_kwargs is None:
        mannwhitney_kwargs = {}
    if wilcoxon_kwargs is None:
        wilcoxon_kwargs = {}
    if multitest_kwargs is None:
        multitest_kwargs = {}

    if test_group not in group_dicts:
        raise ValueError(f"test_group '{test_group}' not found in group_dicts")

    if reference_group is not None and reference_group not in group_dicts:
        raise ValueError(
            f"reference_group '{reference_group}' not found in group_dicts"
        )

    dict_test = group_dicts[test_group]
    dict_ref = group_dicts[reference_group] if reference_group is not None else None

    # --- Helper: extract aggregated 1D values from a single mutation dict ---
    def extract_values_for_mutation(
        rec: dict | None,
        value_key: str,
        subset: list[str],
        agg_func: Callable,
        agg_func_kwargs: dict,
    ) -> list[float]:
        if rec is None or value_key not in rec:
            return []
        value_dict = rec[value_key]
        all_values = []
        for var, vals in value_dict.items():
            if var in subset:
                all_values.append(vals)
        if len(all_values) == 0:
            return []
        arr = np.array(all_values, dtype=float)
        agg_arr = agg_func(arr, **agg_func_kwargs)  # aggregate across variables
        assert agg_arr.ndim == 1, (
            f"Aggregation function {agg_func} returned array with "
            f"{agg_arr.ndim} dimensions"
        )
        return list(agg_arr)

    # --- 1. Compute p-values & basic stats for the test (similar to mutation_mannwhitney_test) ---
    rows_stats = []
    one_sample = dict_ref is None

    if one_sample:
        all_keys = set(dict_test.keys())
    else:
        all_keys = set(dict_test.keys()).union(dict_ref.keys())

    for aa1, aa2 in all_keys:
        rec_a = dict_test.get((aa1, aa2))
        vals_a = extract_values_for_mutation(
            rec_a,
            value_key=value_key,
            subset=subset,
            agg_func=agg_func,
            agg_func_kwargs=agg_func_kwargs,
        )

        if one_sample:
            # test vs zero
            if len(vals_a) == 0:
                stat = np.nan
                p_val = np.nan
                n1 = 0
                n2 = np.nan
            else:
                try:
                    stat, p_val = wilcoxon(
                        vals_a,
                        alternative=alternative,
                        **wilcoxon_kwargs,
                    )
                except Exception:
                    stat = np.nan
                    p_val = np.nan
                n1 = len(vals_a)
                n2 = np.nan
        else:
            rec_b = dict_ref.get((aa1, aa2))
            vals_b = extract_values_for_mutation(
                rec_b,
                value_key=value_key,
                subset=subset,
                agg_func=agg_func,
                agg_func_kwargs=agg_func_kwargs,
            )
            if len(vals_a) == 0 or len(vals_b) == 0:
                stat = np.nan
                p_val = np.nan
            else:
                try:
                    stat, p_val = mannwhitneyu(
                        vals_a,
                        vals_b,
                        alternative=alternative,
                        **mannwhitney_kwargs,
                    )
                except Exception:
                    stat = np.nan
                    p_val = np.nan
            n1 = len(vals_a)
            n2 = len(vals_b)

        rows_stats.append(
            {
                "from_aa": aa1,
                "to_aa": aa2,
                "p_raw": p_val,
                "stat": stat,
                "n1": n1,
                "n2": n2,
            }
        )

    df_stats = pd.DataFrame(rows_stats)

    # Multiple testing correction (optional)
    if multitest:
        pvals = df_stats["p_raw"].values
        _, p_adj, _, _ = multipletests(pvals, method="fdr_bh", **multitest_kwargs)
        df_stats["p_adj"] = p_adj
    else:
        df_stats["p_adj"] = np.nan

    # --- 2. Filter by min_n and sort by p-value ---
    mask_valid = df_stats["n1"] >= min_n
    if not one_sample:
        mask_valid &= df_stats["n2"] >= min_n

    # Drop NaN p-values and insufficient n
    df_stats = df_stats[mask_valid & df_stats["p_raw"].notna()].copy()

    # Sort by p-value ascending
    df_stats.sort_values("p_raw", inplace=True)

    # Optionally keep only top-k mutations
    if max_mutations is not None:
        df_stats = df_stats.head(max_mutations)

    # Build mutation label
    df_stats["mutation"] = df_stats["from_aa"] + "→" + df_stats["to_aa"]

    # Extract the order for plotting
    mutation_order = df_stats["mutation"].tolist()

    # Also build a wide p-value table for reference
    df_p = df_stats.pivot(index="from_aa", columns="to_aa", values="p_raw")

    # --- 3. Build long-format DataFrame with all groups' values for boxplots ---
    long_rows = []

    for (aa1, aa2), row in df_stats.set_index(["from_aa", "to_aa"]).iterrows():
        label = row["mutation"]
        p_raw = row["p_raw"]
        p_adj = row["p_adj"]
        n1 = row["n1"]
        n2 = row["n2"]

        for group_name, gdict in group_dicts.items():
            rec = gdict.get((aa1, aa2))
            vals = extract_values_for_mutation(
                rec,
                value_key=value_key,
                subset=subset,
                agg_func=agg_func,
                agg_func_kwargs=agg_func_kwargs,
            )
            for v in vals:
                long_rows.append(
                    {
                        "from_aa": aa1,
                        "to_aa": aa2,
                        "mutation": label,
                        "group": group_name,
                        "value": v,
                        "p_raw": p_raw,
                        "p_adj": p_adj,
                        "n_test": n1,
                        "n_ref": n2,
                    }
                )

    df_long = pd.DataFrame(long_rows)

    # Ensure categorical order by p-value
    df_long["mutation"] = pd.Categorical(
        df_long["mutation"],
        categories=mutation_order,
        ordered=True,
    )

    # --- 4. Plot ---
    fig, ax = plt.subplots(figsize=figsize)

    sns.boxplot(
        data=df_long,
        x="mutation",
        y="value",
        hue="group",
        palette=palette,
        ax=ax,
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title or f"Mutations sorted by p-value (test={test_group})")

    # Rotate x tick labels for readability
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)

    # Optional: order legend by group_dicts key order
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(
            handles, labels, title="Group", bbox_to_anchor=(1.05, 1), loc="upper left"
        )

    fig.tight_layout()

    return df_long, df_p, fig, ax

In [ ]:
aggregate_counter_ngram3 = compute_aggregate_mutation_counter(
    diff_df=diff_df,
    parent_col="parent",
    mutant_col="mutant",
    ngram_size=3,
    allow_ngrams_overlap=True,
)

mutation_statistics_ngram3 = compute_mutation_statistics_from_df(
    diff_df=diff_df,
    aggregate_counter=aggregate_counter_ngram3,
    parent_col="parent",
    mutant_col="mutant",
    value_cols=[f"{b}_log2_diff" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
    ngram_size=3,
    allow_ngrams_overlap=True,
)

---

In [ ]:
# TODO: boxplots split by group with removal of the mutations for which we do not have smaples
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Build dataframe
df = pd.DataFrame(
    [{"from_aa": k[0], "to_aa": k[1], "count": v} for k, v in transition_counts.items()]
)

# Add a label string
df["label"] = df["from_aa"] + "→" + df["to_aa"]

# Sort
df_sorted = df.sort_values("count", ascending=False)

# Convert label to *ordered* categorical
df_sorted["label"] = pd.Categorical(
    df_sorted["label"],
    categories=df_sorted["label"],  # this keeps original order!
    ordered=True,
)

plt.figure(figsize=(8, 48))
sns.barplot(
    data=df_sorted, x="count", y="label", orient="h"  # now seaborn respects the order
)
plt.xlabel("Count")
plt.ylabel("Mutation")
plt.title("Sorted mutation counts")
plt.tight_layout()
plt.show()

### Subpopulation comparison

In [ ]:
def aa_mutation_heatmap_factory_relative_for_some_preset(
    subset, title, mutation_statistics, mutation_statistics_reference
):
    transition_mean, transition_std, transition_n = aggregate_mutation_dict_to_df(
        mutation_statistics,
        value_key="diff",
        subset=subset,
        agg_func=np.mean,
        agg_func_kwargs={"axis": 0},
    )

    df_p_raw, df_p_adj, df_u, df_n1, df_n2, df_effect = mutation_mannwhitney_test(
        dict_a=mutation_statistics,
        dict_b=mutation_statistics_reference,
        value_key="diff",
        subset=subset,
        agg_func=np.mean,  # matches aggregate_mutation_dict_to_df
        agg_func_kwargs={"axis": 0},
        multitest=True,
    )

    return aa_mutation_heatmap(
        df_mean=transition_mean,
        aa_list=ALL_AA,
        df_std=transition_std,
        df_n=transition_n,
        df_threshold=df_p_adj,
        threshold=0.001,
        threshold_direction="below",
        threshold_color="blue",
        AA_group=AMINO_ACID_CLASSES["polarity"],
        cmap="RdYlGn_r",
        nan_color="white",
        figsize=(24, 24),
        fontsize=11,
        title=title,
        x_label="To AA n-gram",
        y_label="From AA n-gram",
        group_linewidth=4,
        group_colors=group_colors,
        x_bar_gap=0.07,
        y_bar_gap=0.07,
        title_fontsize=30,
        label_fontsize=18,
    )

### Check out different aggregation functions over subset of variables

In [ ]:
(
    gp_transition_mean_agg_median,
    gp_transition_std_agg_median,
    gp_transition_n_agg_median,
) = aggregate_mutation_dict_to_df(
    mutation_statistics_ngram1,
    value_key="diff",
    subset=[f"{b}_log2" for b in GRAM_POSITIVE],
    agg_func=np.median,
)
(
    gn_transition_mean_agg_median,
    gn_transition_std_agg_median,
    gn_transition_n_agg_median,
) = aggregate_mutation_dict_to_df(
    mutation_statistics_ngram1,
    value_key="diff",
    subset=[f"{b}_log2" for b in GRAM_NEGATIVE],
    agg_func=np.median,
)
transition_mean_agg_median, transition_std_agg_median, transition_n_agg_median = (
    aggregate_mutation_dict_to_df(
        mutation_statistics_ngram1,
        value_key="diff",
        subset=[f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
        agg_func=np.median,
    )
)

---

In [ ]:
from collections import defaultdict

# Marginalize over the second variable (sum all counts for each unique second element)
marginal_second = defaultdict(int)
for (first, second), count in transition_counts.items():
    marginal_second[second] += count

# Convert to regular dict and sort by value (descending)
marginal_second = dict(
    sorted(marginal_second.items(), key=lambda x: x[1], reverse=True)
)

marginal_second

---

In [ ]:
df = pd.DataFrame(
    [{"from_aa": k[0], "to_aa": k[1], "count": v} for k, v in transition_counts.items()]
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Build dataframe
df = pd.DataFrame(
    [{"from_aa": k[0], "to_aa": k[1], "count": v} for k, v in transition_counts.items()]
)

# Add a label string
df["label"] = df["from_aa"] + "→" + df["to_aa"]

# Sort
df_sorted = df.sort_values("count", ascending=False)

# Convert label to *ordered* categorical
df_sorted["label"] = pd.Categorical(
    df_sorted["label"],
    categories=df_sorted["label"],  # this keeps original order!
    ordered=True,
)

plt.figure(figsize=(8, 48))
sns.barplot(
    data=df_sorted, x="count", y="label", orient="h"  # now seaborn respects the order
)
plt.xlabel("Count")
plt.ylabel("Mutation")
plt.title("Sorted mutation counts")
plt.tight_layout()
plt.show()